# Análise Global de Mudanças Climáticas com Apache Spark


Este notebook processa dados históricos de temperatura global (Berkeley Earth) e de emissões de CO2 (Our World in Data) usando **PySpark** em modo local (`local[*]`), para identificar tendências de aquecimento, anomalias térmicas e a correlação entre emissões e temperatura.


## 1. Ambiente

O ambiente (Java 17, Python 3.11, virtualenv, dependências) é configurado **fora** deste notebook. Siga o passo a passo completo em **`README.md`**.

Resumo:

1. Instalar versões via `mise install` (ou Java 17 + Python 3.11 manualmente).
2. `python -m venv .venv` e ativar o virtualenv.
3. `pip install -r requirements.txt`.
4. Selecionar o kernel **`.venv`** neste notebook.

Verifique antes de continuar: `python teste_spark.py` deve imprimir a versão do Spark e uma tabela de exemplo.

## 2. Datasets

Os CSVs não estão versionados. Baixe e coloque ambos em `data/`:

- **Temperaturas (Berkeley Earth)** — Kaggle: `GlobalLandTemperaturesByCity.csv` (https://www.kaggle.com/datasets/berkeleyearth/climate-change-earth-surface-temperature-data)
- **Emissões de CO2 (Our World in Data)** — `owid-co2-data.csv` (https://github.com/owid/co2-data)

Caminhos esperados: `data/GlobalLandTemperaturesByCity.csv` e `data/owid-co2-data.csv`.

## 3. Inicializar o Spark

`findspark` localiza a instalação do PySpark e a adiciona ao `sys.path` antes do import. A `SparkSession` é o ponto de entrada da API de DataFrames: aqui roda em modo local usando todos os núcleos (`local[*]`), com memória de driver e executor configuradas e o *Adaptive Query Execution* ativado.

In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("AnaliseClimatica") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark UI disponível em: http://localhost:4040")

Spark UI disponível em: http://localhost:4040


In [4]:
# Verificação rápida do ambiente Spark
print("Versão do Spark:", spark.version)

_dados = [("Brasil", 27.5), ("Canada", -5.2), ("India", 30.1)]
_df = spark.createDataFrame(_dados, ["Pais", "Temp"])
_df.show()

Versão do Spark: 4.1.1
+------+----+
|  Pais|Temp|
+------+----+
|Brasil|27.5|
|Canada|-5.2|
| India|30.1|
+------+----+



In [5]:
## 4. Sanitização de dados

# Leitura dos datasets
df_temp_bruto = spark.read.csv("data/GlobalLandTemperaturesByCity.csv", header=True, inferSchema=True)
df_co2_bruto = spark.read.csv("data/co2_emission.csv", header=True, inferSchema=True)

print("Temperatura colunas:", df_temp_bruto.columns)
print("CO2 colunas:", df_co2_bruto.columns)

Temperatura colunas: ['dt', 'AverageTemperature', 'AverageTemperatureUncertainty', 'City', 'Country', 'Latitude', 'Longitude']
CO2 colunas: ['Entity', 'Code', 'Year', 'Annual CO₂ emissions (tonnes )']


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Limpeza e transformação temp
df_temp_limpo = df_temp_bruto \
    .dropna(subset=["AverageTemperature"]) \
    .withColumn("dt", F.to_date(F.col("dt"))) \
    .withColumn("Ano", F.year("dt")) \
    .withColumn("Mes", F.month("dt")) \
    .filter(F.col("AverageTemperatureUncertainty") < 1.5) \
    .filter((F.col("Ano") >= 1900) & (F.col("Ano") <= 2020))

df_temp_limpo.show(5)

+----------+------------------+-----------------------------+-----+-------+--------+---------+----+---+
|        dt|AverageTemperature|AverageTemperatureUncertainty| City|Country|Latitude|Longitude| Ano|Mes|
+----------+------------------+-----------------------------+-----+-------+--------+---------+----+---+
|1900-01-01|            -0.989|                        0.588|Århus|Denmark|  57.05N|   10.33E|1900|  1|
|1900-02-01|            -2.799|                        0.882|Århus|Denmark|  57.05N|   10.33E|1900|  2|
|1900-03-01|0.5919999999999999|                        0.429|Århus|Denmark|  57.05N|   10.33E|1900|  3|
|1900-04-01|              4.63|          0.41700000000000004|Århus|Denmark|  57.05N|   10.33E|1900|  4|
|1900-05-01|             9.576|                        0.521|Århus|Denmark|  57.05N|   10.33E|1900|  5|
+----------+------------------+-----------------------------+-----+-------+--------+---------+----+---+
only showing top 5 rows


In [ ]:
# Remove agregados
regioes_remover = ["World", "Asia", "Europe", "North America", "South America", "Africa", "European Union (27)"]

# Limpeza, renomea colunas com caracteres que tem problema e filtro de datas
df_co2_limpo = df_co2_bruto \
    .withColumnRenamed("Entity", "Country") \
    .withColumnRenamed("Year", "Ano") \
    .withColumnRenamed("Annual CO₂ emissions (tonnes )", "CO2_Emissions") \
    .filter(~F.col("Country").isin(regioes_remover)) \
    .filter((F.col("Ano") >= 1900) & (F.col("Ano") <= 2020))

df_co2_limpo.show(5)

+-----------+----+----+-------------+
|    Country|Code| Ano|CO2_Emissions|
+-----------+----+----+-------------+
|Afghanistan| AFG|1949|      14656.0|
|Afghanistan| AFG|1950|      84272.0|
|Afghanistan| AFG|1951|      91600.0|
|Afghanistan| AFG|1952|      91600.0|
|Afghanistan| AFG|1953|     106256.0|
+-----------+----+----+-------------+
only showing top 5 rows


In [ ]:
df_temp_anual = df_temp_limpo \
    .groupBy("Country", "Ano") \
    .agg(F.avg("AverageTemperature").alias("TempMediaAnual"))

df_clima_completo = df_temp_anual.join(
    df_co2_limpo,
    ["Country", "Ano"],
    "inner"
)

df_clima_completo.orderBy("Country", "Ano").show(10)

+-----------+----+------------------+----+-------------+
|    Country| Ano|    TempMediaAnual|Code|CO2_Emissions|
+-----------+----+------------------+----+-------------+
|Afghanistan|1949|13.733170212765959| AFG|      14656.0|
|Afghanistan|1950|12.875789473684211| AFG|      84272.0|
|Afghanistan|1951|13.869062499999998| AFG|      91600.0|
|Afghanistan|1952|         14.160375| AFG|      91600.0|
|Afghanistan|1953|14.679905263157893| AFG|     106256.0|
|Afghanistan|1954| 13.57383157894737| AFG|     106256.0|
|Afghanistan|1955| 14.35966666666667| AFG|     153888.0|
|Afghanistan|1956|14.207368421052632| AFG|     183200.0|
|Afghanistan|1957|12.607572916666667| AFG|     293120.0|
|Afghanistan|1958|14.534010416666666| AFG|     329760.0|
+-----------+----+------------------+----+-------------+
only showing top 10 rows


## 5. Próximos passos

Falta fazer:

1. **ETL** — leitura dos CSVs, limpeza (nulos, datas, incerteza, nomes de países), join temperatura × CO2.
2. **As 8 perguntas analíticas** do enunciado (médias móveis, anomalias, correlações, window functions, regressão MLlib).
3. **Visualizações** e artefatos de saída (Parquet).